In [3]:
import pandas as pd
import re
from sklearn.preprocessing import StandardScaler
import statsmodels.formula.api as smf
# Load the uploaded dataset
file_path = 'E:\Economic_Data\Input data\merged-dataset\merged_datasets.csv'
merged_data = pd.read_csv(file_path, parse_dates=['datetime'])

# Load BTC price data
btc_price_file_path = 'E:\Economic_Data\Input data\price.csv'  # Update this path
btc_price_data = pd.read_csv(btc_price_file_path, parse_dates=['datetime'])

# Merge datasets on datetime
merged_data = pd.merge_asof(merged_data.sort_values('datetime'),
                            btc_price_data.sort_values('datetime'),
                            on='datetime',
                            direction='backward')

# Function to clean and convert values to numeric
def clean_and_convert(value):
    if isinstance(value, str):
        # Remove non-numeric characters except for necessary ones like '.' and '-'
        value = re.sub(r'[^\d.-]', '', value)
        # Convert to float
        try:
            return float(value)
        except ValueError:
            return None
    return value

# Apply the function to the 'Actual', 'Forecast', and 'Previous' columns
merged_data['Actual'] = merged_data['Actual'].apply(clean_and_convert)
merged_data['Forecast'] = merged_data['Forecast'].apply(clean_and_convert)
merged_data['Previous'] = merged_data['Previous'].apply(clean_and_convert)

# Calculate differences
merged_data['Actual_Forecast_Diff'] = merged_data['Actual'] - merged_data['Forecast']
merged_data['Actual_Previous_Diff'] = merged_data['Actual'] - merged_data['Previous']

# Fill missing values in the differences with zero
merged_data['Actual_Forecast_Diff'].fillna(0, inplace=True)
merged_data['Actual_Previous_Diff'].fillna(0, inplace=True)

# Adding an identifier for the events
merged_data['Event_ID'] = merged_data['Event'].factorize()[0]

# Scale the explanatory variables
scaler = StandardScaler()
merged_data[['Actual_Forecast_Diff', 'Actual_Previous_Diff']] = scaler.fit_transform(merged_data[['Actual_Forecast_Diff', 'Actual_Previous_Diff']])

# Ensure the percentage change columns are correctly calculated
time_intervals = [5, 15, 30, 60]

for interval in time_intervals:
    merged_data[f'Close_{interval}min'] = merged_data['close'].shift(-interval)
    merged_data[f'Pct_Change_{interval}min'] = ((merged_data[f'Close_{interval}min'] - merged_data['close']) / merged_data['close']) * 100

# Remove rows with NaNs in the percentage change columns
merged_data.dropna(subset=[f'Pct_Change_{interval}min' for interval in time_intervals], inplace=True)

# Simplified Hierarchical Model Analysis
results = {}

for interval in time_intervals:
    formula = f'Pct_Change_{interval}min ~ Actual_Forecast_Diff + Actual_Previous_Diff'
    model = smf.mixedlm(formula, data=merged_data, groups=merged_data["Event_ID"], re_formula="~1")
    result = model.fit()
    results[interval] = result

# Display results
summary_results = {}
for interval, result in results.items():
    summary_results[interval] = result.summary()

# Create a matrix to summarize the impact
impact_matrix = pd.DataFrame(index=merged_data['Event'].unique(), columns=[f'Impact_{interval}min_Actual_Forecast' for interval in time_intervals] + [f'Impact_{interval}min_Actual_Previous' for interval in time_intervals])

for interval, result in results.items():
    params = result.params
    impact_matrix[f'Impact_{interval}min_Actual_Forecast'] = params['Actual_Forecast_Diff']
    impact_matrix[f'Impact_{interval}min_Actual_Previous'] = params['Actual_Previous_Diff']


impact_matrix.head(), summary_results



C:\Users\Zeinab\AppData\Local\Temp\ipykernel_8928\2289062267.py:41: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  merged_data['Actual_Forecast_Diff'].fillna(0, inplace=True)
C:\Users\Zeinab\AppData\Local\Temp\ipykernel_8928\2289062267.py:42: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a c

(                                       Impact_5min_Actual_Forecast  \
 S&P Global US Manufacturing PMI (Dec)                    -0.011007   
 Construction Spending (MoM) (Nov)                        -0.011007   
 3-Month Bill Auction                                     -0.011007   
 6-Month Bill Auction                                     -0.011007   
 Mortgage Market Index                                    -0.011007   
 
                                        Impact_15min_Actual_Forecast  \
 S&P Global US Manufacturing PMI (Dec)                     -0.001606   
 Construction Spending (MoM) (Nov)                         -0.001606   
 3-Month Bill Auction                                      -0.001606   
 6-Month Bill Auction                                      -0.001606   
 Mortgage Market Index                                     -0.001606   
 
                                        Impact_30min_Actual_Forecast  \
 S&P Global US Manufacturing PMI (Dec)                      0.0660